# 06b — Step 6c: region metrics, thicker target, schedule, FiLM

**What this notebook does.** `06_train.ipynb`'s placement/thickness
decomposition, corrected and re-run on two independent checkpoints, found the
`dev` fold's real uhcs2 failure is **MISPLACED** (skeleton Dice 0.057 BELOW
pixel Dice 0.149 -- the model's predicted centrelines do not spatially
coincide with the true ones), not the earlier OVER-DETECTION verdict its
calibration set could not actually distinguish from misplacement. This
notebook scores four things against that finding, ONE AT A TIME, on the SAME
held-out validation tiles:

| arm | run name | what changes vs. the previous arm |
| --- | --- | --- |
| A | `dev` | nothing -- adds region-level metrics to the EXISTING `best.pt`, no retraining |
| B | `dev-w4` | `boundary_gt.line_width_px` 2 -> 4 (GT + manifests + fold_stats regenerated first) |
| C | `dev-w4-sched` | + `encoder_lr_scale` 0.3, `min_lr_scale` 0.05, `epochs` 80, early stopping (`patience` 10) |
| D | `dev-w4-sched-film` | + `model.film.enabled: true` (per-dataset FiLM conditioning) |

Set `ARM` in the settings cell and re-run this notebook top to bottom -- one
arm per run, same as `06_train.ipynb` runs one `FOLD` at a time. The final
cell compares whichever arms have a pushed report so far.

**What must already exist.**

- Everything `06_train.ipynb` needs, plus `reports/train_dev_<platform>.json`
  with a `best.pt` under `PERSISTENT_DIR/checkpoints/dev/` (arm A evaluates it).
- For arms B/C/D: `02_boundary_gt.ipynb` re-run with `line_width_px=4`
  (Colab only) and `03_tiling.ipynb` re-run afterwards, so
  `reports/gt_extraction.json` and `configs/fold_stats.yaml` reflect the wider
  target. This notebook checks that and refuses to train otherwise --
  regenerating the ground truth is steps 2 and 3's job, not this notebook's.

**What it produces.** `reports/region_metrics_<run>_<platform>.{json,md}` for
every arm; `reports/train_<run>_<platform>.{json,md}` for arms B/C/D (arm A's
already exists); `reports/film_inference_<run>_<platform>.{json,md}` for arm D
only. All pushed back to the repo, keyed by run name and host exactly as
`06_train.ipynb`'s reports are.

**Expected runtime on a free T4.** Arm A: a few minutes (validation-only, no
training). Arms B/C: comparable to `06_train.ipynb`'s dev run (~2.5 min/epoch,
up to 80 epochs for C, likely far fewer with early stopping). Arm D: similar
to C, plus `len(vocabulary) + 1` extra validation passes for the FiLM
inference-mode comparison at the end (a few extra minutes).

## Cell 1 — the standard bootstrap block

Identical to every other notebook's cell 1. Reads `GH_TOKEN` from the host
secret store, fetches `scripts/bootstrap_session.py`, then hands over to
`bootstrap()`, which syncs the repo, installs what is missing, mounts Drive on
Colab and returns `PATHS`.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Cell 2 — Session support, and the clock

Same dispatch point `06_train.ipynb` uses: `session.for_host(PATHS)` is the
single place that knows which host this is, and `session.guard()` stops a
training arm on an epoch boundary before a time-limited host kills the kernel.
Arm A never trains, but calling this here anyway keeps every arm's cells
identical rather than branching cell 2 on `ARM`.

In [ ]:
from src import session as session_mod

session = session_mod.for_host(PATHS)

print(f"host support : {type(session).__name__} (platform {session.platform})")
print(f"persists     : {session.persists}   time-limited: {session.time_limited}")
print()
print(session.describe())

## Cell 3 — Pick the arm, resolve its settings

**Set `ARM` below and re-run the notebook for the next one.** Each arm's
settings are built as an override on top of `configs/default.yaml` -- the same
dict-override pattern `06_train.ipynb` uses for `train.exclude_datasets` --
rather than editing the shared config file, so `dev`'s already-committed
report stays reproducible from that file unchanged and every other arm is
unaffected by running this one.

Arms B/C/D need `boundary_gt.line_width_px=4` ground truth to already exist:
this cell reads `reports/gt_extraction.json`'s recorded setting and REFUSES to
proceed if it does not match, naming the notebooks to re-run first. This is a
read of what steps 2 and 3 actually produced, not an assumption that they were
run correctly.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src import dataset as ds
from src import losses as losses_mod
from src import model as model_mod
from src import train as train_mod

FOLD = "dev"
ARM = "A"          # "A" | "B" | "C" | "D" -- change this and re-run for the next arm


def progress(seq, desc=""):
    return tqdm(seq, desc=desc, leave=False)


ARM_RUN_NAMES = {"A": "dev", "B": "dev-w4", "C": "dev-w4-sched",
                "D": "dev-w4-sched-film"}
if ARM not in ARM_RUN_NAMES:
    raise ValueError(f"ARM must be one of {sorted(ARM_RUN_NAMES)}, got {ARM!r}")
RUN_NAME = ARM_RUN_NAMES[ARM]
NEEDS_TRAINING = ARM != "A"
NEEDS_WIDTH_4 = ARM in ("B", "C", "D")

train_settings = train_mod.load_config()
model_settings = model_mod.load_config()
loss_settings = losses_mod.load_config()
ds_settings = ds.load_config()

if NEEDS_WIDTH_4:
    gt_report_path = Path(PATHS["reports_dir"]) / "gt_extraction.json"
    if not gt_report_path.is_file():
        raise train_mod.TrainError(
            f"arm {ARM} needs boundary_gt.line_width_px=4, but "
            f"{gt_report_path} does not exist -- run 02_boundary_gt.ipynb "
            "with that override first (Colab only), then 03_tiling.ipynb.")
    recorded_width = json.loads(gt_report_path.read_text())["settings"]["line_width_px"]
    if recorded_width != 4:
        raise train_mod.TrainError(
            f"arm {ARM} needs boundary_gt.line_width_px=4, but "
            f"{gt_report_path} was written under line_width_px={recorded_width}. "
            "Re-run 02_boundary_gt.ipynb with that override, then "
            "03_tiling.ipynb to regenerate manifests and "
            "configs/fold_stats.yaml, before training this arm. Results "
            "before/after this change are NOT directly comparable: "
            "pos_weight and every boundary fraction move with it.")
    print(f"boundary_gt.line_width_px = {recorded_width} confirmed "
          f"(from {gt_report_path})")

if ARM in ("C", "D"):
    train_settings = dict(train_settings)
    train_settings.update(encoder_lr_scale=0.3, min_lr_scale=0.05,
                          epochs=80, patience=10)
if ARM == "D":
    model_settings = dict(model_settings)
    model_settings["film"] = {"enabled": True, "embed_dim": 16}

fold_stats = ds.load_fold_stats()
entry = fold_stats["folds"][FOLD]

trainer = train_mod.Trainer(fold=FOLD, resolved=PATHS, settings=train_settings,
                            model_settings=model_settings,
                            loss_settings=loss_settings,
                            dataset_settings=ds_settings, run_name=RUN_NAME)

print(f"arm {ARM}: run_name={trainer.run_name}  fold={FOLD}  "
      f"held out {trainer.held_out}")
print(f"  needs training: {NEEDS_TRAINING}")
print(f"  encoder_lr_scale {train_settings['encoder_lr_scale']}  "
      f"min_lr_scale {train_settings['min_lr_scale']}  "
      f"epochs {train_settings['epochs']}  patience {train_settings['patience']}")
print(f"  model.film.enabled {model_settings['film']['enabled']}")
print(f"  config hash {trainer.hash}")
if trainer.film_enabled:
    print(f"  FiLM vocabulary: {trainer.film_vocabulary}")

## Cell 4 — Train (arms B/C/D), or skip (arm A)

Arm A scores the checkpoint `06_train.ipynb` already produced and pushed --
**no retraining**, per the instruction to measure the current model on its
real objective before changing anything. `trainer.setup()` still runs for
every arm because it is what builds `trainer.val_ds`, used by every cell
after this one regardless of arm.

For B/C/D, this is the same resumable, checkpoint-after-every-epoch loop as
`06_train.ipynb`'s Cell 8, with one addition: a `stopped_early` flag on the
final history record when `train.patience` triggers. That is a clean,
expected stop (the run reached a plateau), not the same thing as
`SessionStopped` (the HOST ran out of time) -- both are handled, and the
printed message says which one happened.

In [ ]:
summary = trainer.setup(progress=progress)
print(f"device {summary['device']}  batch {summary['batch_size']}  "
      f"workers {summary['num_workers']}")
print(f"train composition {summary['train_composition']}")
print(f"val composition   {summary['val_composition']}")

if NEEDS_TRAINING:
    status = trainer.maybe_resume()
    print(f"\nresume: {status['reason']}")
    if status["resumed"]:
        print(f"continuing from epoch {status['start_epoch']}")

    def on_epoch_end(record, trainer):
        flag = " <<< BEST" if record["is_best"] else ""
        stop_flag = "  STOPPING (patience)" if record.get("stopped_early") else ""
        print(f"epoch {record['epoch']:>3}  {record['seconds']:.1f}s  "
              f"lr {record['lr']:.3e}  score {record['score']:.4f} "
              f"({record['score_key']}){flag}{stop_flag}")

    started = time.perf_counter()
    session_stopped = False
    try:
        trainer.fit(on_epoch_end=session.guard(on_epoch_end), progress=progress)
    except session_mod.SessionStopped as stop:
        session_stopped = True
        print("\n" + "=" * 72)
        print("SESSION BUDGET REACHED -- this is a clean stop, not an error")
        print("=" * 72)
        print(stop)

    print(f"\n{'stopped on session budget' if session_stopped else 'loop finished'}: "
          f"{len(trainer.history)} epoch(s) in trainer.history, "
          f"{(time.perf_counter() - started) / 60:.1f} min this session")
    if trainer.history and trainer.history[-1].get("stopped_early"):
        print(f"EARLY STOPPING (patience={train_settings['patience']}): no new "
              f"best-threshold Dice for "
              f"{trainer.history[-1]['epochs_since_improvement']} epochs")
    print(f"best epoch {trainer.best['epoch']}: "
          f"{trainer.best['key']} Dice {trainer.best['metric']:.4f} "
          f"at threshold {trainer.best['threshold']}")

    print()
    session.survival_report(trainer.run_name, trainer=trainer)
else:
    print(f"\nARM A: no training -- scoring the EXISTING checkpoint at "
          f"{trainer.best_path}")
    if not trainer.best_path.is_file():
        raise train_mod.TrainError(
            f"no checkpoint at {trainer.best_path}; run 06_train.ipynb on "
            f"fold {FOLD!r} at least once before running arm A here.")

## Cell 5 — Region-level metrics, this arm's `best.pt`

Loaded fresh from `best.pt` via `load_checkpoint_model` (the same fold/hash
guards `maybe_resume` applies), scored on `trainer.val_ds`, never on whatever
is left in `trainer.model` after `fit()` -- that could be the LAST epoch, not
the BEST one. `evaluate_region_metrics` adds marker-controlled watershed vs.
the region partition the ground truth implies (ARI, VI, Panoptic Quality,
over-segmentation factor) to the pixel/skeleton decomposition
`06_train.ipynb` already reports, so a change that moves pixel Dice without
moving region quality is visible as such.

In [ ]:
if not trainer.best_path.is_file():
    raise train_mod.TrainError(f"no checkpoint at {trainer.best_path} to evaluate.")

eval_model, best_state = train_mod.load_checkpoint_model(
    trainer.best_path, model_settings, fold=FOLD, expected_hash=trainer.hash,
    device=trainer.device,
    film_vocabulary=trainer.film_vocabulary if trainer.film_enabled else None)

print(f"loaded {trainer.best_path}")
print(f"  epoch {best_state['epoch']}  held out {best_state.get('held_out')}")
print(f"  selected on: {best_state['best']['criterion']}  "
      f"({best_state['best']['key']} = {best_state['best']['metric']:.4f} "
      f"at threshold {best_state['best']['threshold']})")

thresholds = train_mod.best_epoch_thresholds(best_state)
print(f"\nper-dataset thresholds used below (from the best epoch's own sweep): "
      f"{thresholds}")

region_settings = train_settings["region_metrics"]
val_batch = int(train_settings["val_batch_size"] or train_settings["batch_size"])

region_results = train_mod.evaluate_region_metrics(
    eval_model, trainer.val_ds, thresholds, device=trainer.device,
    marker_threshold=region_settings["watershed_marker_threshold"],
    amp_enabled=trainer.amp_enabled, batch_size=val_batch, num_workers=0)

print("\nregion-level metrics, per dataset (mean over validation tiles):")
print(pd.DataFrame(region_results).T.to_string(float_format=lambda v: f"{v:.4f}"))

region_md, region_json = train_mod.write_region_metrics_report(
    trainer.run_name, trainer.platform, region_results,
    marker_threshold=region_settings["watershed_marker_threshold"],
    reports_dir=Path(PATHS["reports_dir"]))
print(f"\nwrote {region_md}")
print(f"wrote {region_json}")

DILATIONS = (1, 2, 3)
DISTANCES = (1, 2, 3, 5)
decomposition = train_mod.decompose_error(
    eval_model, trainer.val_ds, thresholds, device=trainer.device,
    amp_enabled=trainer.amp_enabled, dilations=DILATIONS, distances=DISTANCES,
    batch_size=val_batch, num_workers=0)

held = decomposition.get(trainer.held_out)
if held is not None:
    print(f"\n{trainer.held_out} (held out) decomposition verdict: {held['label']}")
    print(f"  pixel Dice {held['pixel_dice']:.4f}  skeleton Dice "
          f"{held['skeleton_dice']:.4f}  ({held['verdict']})")

## Cell 6 — FiLM inference-mode comparison (arm D only)

The held-out dataset has no FiLM embedding of its own -- it was never trained
on. `evaluate_film_inference_modes` scores it under all three ways to
condition it anyway: the mean of the training embeddings (what ordinary
validation already does automatically), each training dataset's own embedding
in turn, and whichever of those gives a predicted boundary fraction closest to
the held-out dataset's true one. Each row carries the full decomposition and
region metrics, not just pixel Dice, so a mode that only moves Dice without
moving the MISPLACED verdict is visible as such.

In [ ]:
if ARM == "D":
    film_results = train_mod.evaluate_film_inference_modes(
        eval_model, trainer.val_ds, thresholds, device=trainer.device,
        vocabulary=trainer.film_vocabulary, fold_entry=entry,
        held_out=trainer.held_out, amp_enabled=trainer.amp_enabled,
        region_marker_threshold=region_settings["watershed_marker_threshold"],
        dilations=DILATIONS, distances=DISTANCES,
        batch_size=val_batch, num_workers=0)

    rows = []
    for mode_name, row in film_results.items():
        dec = row["decomposition"]
        reg = row["region_metrics"] or {}
        rows.append({"mode": mode_name, "pixel_dice": dec["pixel_dice"],
                    "verdict": dec["label"], "ari": reg.get("ari"),
                    "vi": reg.get("vi"), "pq": reg.get("pq"),
                    "pred_fraction": row.get("pred_fraction")})
    print(f"FiLM inference modes on the held-out dataset ({trainer.held_out}):")
    print(pd.DataFrame(rows).set_index("mode")
         .to_string(float_format=lambda v: f"{v:.4f}"))
    print(f"\nclosest chose: {film_results['closest']['chosen_dataset']} "
          f"(target true boundary fraction "
          f"{film_results['closest']['target_true_fraction']:.4f})")

    film_md, film_json = train_mod.write_film_inference_report(
        trainer.run_name, trainer.platform, trainer.held_out, film_results,
        reports_dir=Path(PATHS["reports_dir"]))
    print(f"\nwrote {film_md}")
    print(f"wrote {film_json}")
else:
    print(f"ARM {ARM}: FiLM inference-mode comparison only applies to arm D "
          "(model.film.enabled).")

## Cell 7 — Checks

Where this arm is declared correct or not.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


check("run_name matches this arm's naming convention",
      trainer.run_name == RUN_NAME, trainer.run_name)
check("best checkpoint exists", trainer.best_path.is_file(), str(trainer.best_path))

val_datasets = {r["dataset"] for r in trainer.val_ds.rows}
check("region metrics were computed for every validation dataset",
      set(region_results) == val_datasets,
      f"{sorted(region_results)} vs {sorted(val_datasets)}")
for name, row in region_results.items():
    check(f"region metrics for {name} are in range",
          0.0 <= row["ari"] <= 1.0 and row["vi"] >= -1e-9
          and 0.0 <= row["pq"] <= 1.0 and row["over_segmentation_factor"] > 0,
          f"ari {row['ari']:.3f}  vi {row['vi']:.3f}  pq {row['pq']:.3f}  "
          f"over-seg {row['over_segmentation_factor']:.2f}")
check("region metrics report was written", region_json.is_file(), str(region_json))

if ARM == "D":
    check("film_inference report was written", film_json.is_file(), str(film_json))
    check("closest mode chose one of the training datasets",
          film_results["closest"]["chosen_dataset"] in trainer.film_vocabulary,
          film_results["closest"]["chosen_dataset"])
    check("every FiLM mode reports a decomposition label for the held-out dataset",
          all(row["decomposition"].get("label") for row in film_results.values()),
          {name: row["decomposition"]["label"] for name, row in film_results.items()})

if NEEDS_TRAINING:
    check("at least one epoch completed", len(trainer.history) >= 1,
          f"{len(trainer.history)} epoch(s)")
    if train_settings.get("patience") is not None:
        check("early stopping never fires before patience epochs without improvement",
              all(not h.get("stopped_early")
                  or h["epochs_since_improvement"] >= train_settings["patience"]
                  for h in trainer.history),
              f"patience={train_settings['patience']}")

print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed")
if not all(ok for _, ok in checks):
    raise train_mod.TrainError("one or more checks failed; see above")

## Cell 8 — Write the training report (B/C/D), push everything

Arm A never trains, so there is no new `train_<fold>.json` to write -- only
the region-metrics report is new for it. Only `reports/` and `configs/` are
pushed; checkpoints are never committed, exactly as in `06_train.ipynb`.

In [ ]:
from scripts.push_results import push_results

expect = [region_md, region_json]
if NEEDS_TRAINING:
    train_md, train_json = train_mod.write_report(trainer)
    print(f"wrote {train_md}")
    print(f"wrote {train_json}")
    expect += [train_md, train_json]
if ARM == "D":
    expect += [film_md, film_json]

message = (f"step 6c: arm {ARM} ({trainer.run_name}) -- region metrics"
          + (", training" if NEEDS_TRAINING else "")
          + (", FiLM inference modes" if ARM == "D" else ""))
pushed = push_results(message, paths=PATHS, expect=expect)
print(f"\npushed to origin/{PATHS['branch']}: {pushed}")

## Cell 9 — Arms compared, so far

Whichever of A/B/C/D have a pushed report at the time this cell runs, side by
side on the SAME held-out validation tiles, plus a config diff between
consecutive arms so the run-order requirement (one variable at a time) is
checked rather than assumed. Re-run this cell (or the whole notebook) after
each new arm finishes to extend the table.

In [ ]:
reports = train_mod.load_run_reports(FOLD, reports_dir=Path(PATHS["reports_dir"]))
present = sorted({run for run, _ in reports})
print(f"arms with a pushed training report so far: {present}")
missing = [name for name in ARM_RUN_NAMES.values() if name not in present]
if missing:
    print(f"missing: {missing} -- set ARM to the corresponding letter and "
          "re-run this notebook to fill them in")

comparison = train_mod.compare_runs(reports, entry["held_out"], row="best")
if comparison["runs"]:
    print(f"\n{entry['held_out']} (HELD OUT) at each arm's own tuned "
          "threshold, on the SAME validation tiles:")
    print(pd.DataFrame(comparison["runs"]).T
         .to_string(float_format=lambda v: f"{v:.4f}"))
    print(f"\n{comparison['note']}")

# Region-level reports, side by side -- read alongside pixel Dice above: a
# change that raises Dice without raising ARI/PQ or lowering VI has not
# actually improved what the downstream watershed stage will see.
for run_name in ARM_RUN_NAMES.values():
    region_report_path = (Path(PATHS["reports_dir"])
                          / f"region_metrics_{run_name}_{PATHS['platform']}.json")
    if region_report_path.is_file():
        payload = json.loads(region_report_path.read_text())
        held_row = payload["results"].get(entry["held_out"])
        if held_row:
            print(f"\n{run_name}: {entry['held_out']} region metrics -- "
                  f"ARI {held_row['ari']:.3f}  VI {held_row['vi']:.3f}  "
                  f"PQ {held_row['pq']:.3f}  "
                  f"over-seg {held_row['over_segmentation_factor']:.2f}")

# Config differences between CONSECUTIVE arms only -- the run-order
# requirement is that each arm changes exactly one thing versus the one
# before it, not versus the original baseline.
ordered = [name for name in ARM_RUN_NAMES.values() if name in present]
for left, right in zip(ordered, ordered[1:]):
    left_key = next(k for k in reports if k[0] == left)
    right_key = next(k for k in reports if k[0] == right)
    print(f"\nconfig difference {left} -> {right}:")
    for line in train_mod.diff_runs(reports, left_key, right_key):
        print(f"  {line}")

if len(ordered) == len(ARM_RUN_NAMES):
    print("\nAll four arms are present. Read the config-hash diffs above "
          "together with each arm's own decomposition verdict (printed by "
          "Cell 5's run for that arm) to say whether D's change, if any, "
          "moved the MISPLACED label itself or only its magnitude -- that "
          "distinction is the point of running these one variable at a time.")